In [34]:
import os

import pandas as pd
from openai import OpenAI
from pandas import DataFrame
import tiktoken
import numpy as np
from tqdm import tqdm

tqdm.pandas()
openai_base_url = os.getenv("OPENAI_BASE_URL")
openai_api_key = os.getenv("OPENAI_API_KEY")

In [36]:
# 1. 数据预处理
df: DataFrame = pd.read_csv("datasets/fine_food_reviews_1k.csv")
df = df[['Time', 'ProductId', 'UserId', 'Score', 'Summary', 'Text']]

# 删除任一一列存在NaN,NaT的行
df = df.dropna()
# 组合特征
df['combined'] = df.apply(lambda row: f"Title:{row['Summary']}; Content:{row['Text']}", axis=1)
df = df.sort_values('Time')
df.drop('Time', axis=1, inplace=True)
df.head(2)

,ProductId,UserId,Score,Summary,Text,combined
0,B003XPF9BO,A3R7JR3FMEBXQB,5,where does one start...and stop... with a tre...,Wanted to save some to bring to my Chicago fam...,Title:where does one start...and stop... with...
1,B003JK537S,A3JBPC3WFUT5ZP,1,Arrived in pieces,"Not pleased at all. When I opened the box, mos...",Title:Arrived in pieces; Content:Not pleased a...


In [37]:
# 2. 生成Embedding之后的向量空间，并且保存
embedding_model = 'Qwen/Qwen3-Embedding-8B'

# 参数
max_tokens = 8000
top_n = 1000

# 分词器计算token
tokenizer_name = 'cl100k_base'
tokenizer = tiktoken.get_encoding(encoding_name=tokenizer_name)
# 计算token数量
df['count_token'] = df['combined'].apply(lambda x: len(tokenizer.encode(x)))
# 判断token的数量不能超过官方的阈值
df = df[df['count_token'] <= max_tokens].tail(top_n)
len(df)

1000

In [38]:
client = OpenAI(
    base_url=openai_base_url,
    api_key=openai_api_key,
)


def embedding_text(text, model):
    """
    通过Embedding模型处理文本数据
    :param text: 需要处理的文本
    :param model: 嵌入模型
    :return: 文本向量
    """
    resp = client.embeddings.create(input=[text], model=model)
    return resp.data[0].embedding

In [39]:
df['embedding'] = df['combined'].progress_apply(lambda x: embedding_text(x, embedding_model))
df.head(1)['embedding']

 30%|███       | 301/1000 [15:23<35:44,  3.07s/it]  


APIConnectionError: Connection error.

In [ ]:
df.to_csv('output/embedding_1000.csv')

In [ ]:
#  使用 T-SEN 可视化嵌入结果
from matplotlib import pyplot as plt
import matplotlib
from sklearn.manifold import TSNE

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False


In [ ]:
tsen = TSNE(n_components=2, perplexity=15, random_state=42, init='random', learning_rate=2)
# 变成二维数组
matrix = np.vstack(df['embedding'].values)

# 使用t-sne模型降维，的要一个二维的参数
matrix_2d = tsen.fit_transform(matrix)

matrix_2d

In [ ]:
# 可视化，根据不同颜色来区分不同的评分
colors = ["red", "darkorange", "gold", "turquoise", "darkgreen"]

x = matrix_2d[:, 0]
y = matrix_2d[:, 1]

colors_indexes = df['Score'].values - 1
color_map = matplotlib.colors.ListedColormap(colors)

plt.scatter(x=x, y=y, c=colors_indexes, cmap=color_map, alpha=0.6)

plt.title('使用T-SNE降维后的评论')
plt.show()

上图多个群落代表文本中不同的情绪或者情感或者观点，颜色代表不同的评分，可以看出不同类型的的评论的评分分布

In [ ]:
# 使用KMeans聚类算法可视化结果
from sklearn.cluster import KMeans

# 3个类别
km = KMeans(6, init='k-means++', random_state=43, n_init=10)
km.fit(matrix)
# 表示0 1 2 表示三个类别
df['kmeans_label'] = km.labels_
colors = ["red", "darkorange", "gold", "turquoise", "darkgreen", "blue"]
color_map = matplotlib.colors.ListedColormap(colors)
colors_indexes = df['kmeans_label'].values
plt.scatter(x=x, y=y, c=colors_indexes, cmap=color_map, alpha=0.6)
plt.title('使用聚类和T-SNE降维后的评论')
plt.show()

上图可以看出，不同的类别的评论，颜色代表不同的类别，颜色越接近的，代表相似的